In [1]:
import os
import requests

In [2]:
# Download the training text to local file
if not os.path.exists("the-verdict.txt"):
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    file_path = "the-verdict.txt"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [3]:
# Read the training text
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f"Total number of characters: {len(raw_text)}")
print(raw_text[:80])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow en


In [4]:
# Cleat test: use regex to split on punctuation and white-space
import re
text = "Hello, world. This, is a test."
result = re.split(r'([,.:;?_!\"()\']|--|\s)', text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [5]:
# Clean training text
preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(f"Total number of words: {len(preprocessed)}")
print(preprocessed[:30])

Total number of words: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [6]:
# Compile sorted unique word list
all_tokens = sorted(list(set(preprocessed)))

# Add special tokens for end-of-text, unknown word
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab_size = len(all_tokens)
print(f"Total number of unique tokens: {vocab_size}")

Total number of unique tokens: 1132


In [7]:
# Create a vocabulary dictionary
vocab = { token:integer for integer,token in enumerate(all_tokens) }
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 30:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)


In [8]:
# Define vocabulary encoder/decoder
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # Setup word to numeric dictionary
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items() }

    def encode(self, text):
        # Encode text into numeric vector
        preprocessed = re.split(r'([,.:;?_!\"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]

        # Handle unknown words
        preprocessed = [ item if item in self.str_to_int else "<|unk|>" for item in preprocessed ]

        # Convert words to numeric vector
        ids = [ self.str_to_int[s] for s in preprocessed ]
        return ids

    def decode(self, ids):
        # Decode numeric vector into text
        text = " ".join([self.int_to_str[i] for i in ids])
        
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!\"()\'])', r'\1', text)
        return text

In [9]:
# Test vocabulary encoder
tokenizer = SimpleTokenizerV2(vocab)
text = "\"It's the last he painted, you know,\" Mrs. Gisburn said with pardonable pride."
encoded_ids = tokenizer.encode(text)
print(encoded_ids)

# Test vocabulary decoder
decoded_text = tokenizer.decode(tokenizer.encode(text))
print(decoded_text)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [10]:
# Test vocabulary encoder
tokenizer = SimpleTokenizerV2(vocab)
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
encoded_ids = tokenizer.encode(text)
print(encoded_ids)

# Test vocabulary decoder
decoded_text = tokenizer.decode(tokenizer.encode(text))
print(decoded_text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [11]:
# Set standard tokenizer
from importlib.metadata import version
import tiktoken
print(f"tiktoken version: {version('tiktoken')}")

tiktoken version: 0.12.0


In [12]:
# Test GPT-2 encoder
tokenizer = tiktoken.get_encoding("gpt2")
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
encoded_ids = tokenizer.encode(text, allowed_special={ "<|endoftext|>" })
print(encoded_ids)

# Test GPT-2 decoder
decoded_text = tokenizer.decode(tokenizer.encode(text, allowed_special={ "<|endoftext|>" }))
print(decoded_text)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [13]:
# Read the training text
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print(f"Total number of characters: {len(raw_text)}")

Total number of characters: 20479


In [14]:
# Encode training text
enc_text = tokenizer.encode(raw_text)
print(f"Total number of characters: {len(enc_text)}")

Total number of characters: 5145


In [15]:
# Create text chunk
enc_sample = enc_text[50:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}\ny: {y}")

x: [290, 4920, 2241, 287]
y: [4920, 2241, 287, 257]


In [16]:
# Show encoded prediction
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(context, "->", desired)

[290] -> 4920
[290, 4920] -> 2241
[290, 4920, 2241] -> 287
[290, 4920, 2241, 287] -> 257


In [17]:
# Show decoded prediction
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "->", tokenizer.decode([desired]))

 and ->  established
 and established ->  himself
 and established himself ->  in
 and established himself in ->  a


In [18]:
# Install PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")

from torch.utils.data import Dataset, DataLoader

PyTorch version: 2.9.1


In [19]:
# Class to load dataset end GPT-2 encode
class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    # Tokenize the entire text
    token_ids = tokenizer.encode(txt, allowed_special={ "<|endoftext|>" })
    assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

    # Use a sliding window to chunk the book into overlapping sequences of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i + max_length]
      target_chunk = token_ids[i + 1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [20]:
# Function to create encoding using PyTorch dataloader.
# Dataloader provides utilities for input data batching and shuffling.
def create_dataloader_v1(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

  # Initialize the tokenizer
  tokenizer = tiktoken.get_encoding("gpt2")

  # Create dataset
  dataset = GPTDatasetV1(text, tokenizer, max_length, stride)

  # Create iterable dataloader to load batches of data:
  # - batch_size loads batch of data of specified size during each iteration
  # - shuffle=True changes the order of data indices to achieve better generalization
  # - drop_last=True drops the last batch if it is shorter than the batch_size to prevent loss spikes during training
  # - num_workers does not work in notebooks
  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
  return dataloader

In [21]:
# Encode the data in batch of size 1 (just to visualize it working)
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)

# Print first batch
inputs, targets = next(data_iter)
print("batch 1")
print("inputs", inputs)
print("targets", targets)

batch 1
inputs tensor([[  40,  367, 2885, 1464]])
targets tensor([[ 367, 2885, 1464, 1807]])


In [22]:
# Print second batch
inputs, targets = next(data_iter)
print("batch 2")
print("inputs", inputs)
print("targets", targets)

batch 2
inputs tensor([[ 367, 2885, 1464, 1807]])
targets tensor([[2885, 1464, 1807, 3619]])


In [23]:
# Create data loader for batch of size 8 of 4 tokens (more realistic)
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)

# Print first batch
inputs, targets = next(data_iter)
print("batch 1")
print("inputs", inputs)
print("targets", targets)

batch 1
inputs tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [24]:
# Print second batch
inputs, targets = next(data_iter)
print("batch 2")
print("inputs", inputs)
print("targets", targets)

batch 2
inputs tensor([[  287,   262,  6001,   286],
        [  465, 13476,    11,   339],
        [  550,  5710,   465, 12036],
        [   11,  6405,   257,  5527],
        [27075,    11,   290,  4920],
        [ 2241,   287,   257,  4489],
        [   64,   319,   262, 34686],
        [41976,    13,   357, 10915]])
targets tensor([[  262,  6001,   286,   465],
        [13476,    11,   339,   550],
        [ 5710,   465, 12036,    11],
        [ 6405,   257,  5527, 27075],
        [   11,   290,  4920,  2241],
        [  287,   257,  4489,    64],
        [  319,   262, 34686, 41976],
        [   13,   357, 10915,   314]])


**Embedding Layer**

The purpose of the embedding layer is to convert the integer token IDs into floating-point numbers that can be optimized in later logic.

It is not important what the small random values of the embedding layer are at this point. The values will get trained in later logic.

In [25]:
# Create an embedding layer of size 6x3
vocab_size = 6
output_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# Show the 6x3 weight matrix of small random values
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [26]:
# Demonstrate one-hot encoding vector applied to embedding layer
input_ids = torch.tensor([2, 3, 5, 1])
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [27]:
# Create an 256-dimensional embedding layer for GPT-3 tokenizer vocabulary
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [28]:
# Create data loader for batch size 8 of 4 tokens
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)

# Print first batch of token IDs
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("batch 1")
print("inputs", inputs)
print("targets", targets)

batch 1
inputs tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [29]:
# Sample a batch of text using the embedding layer and show size 8x4x256
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [ ]:
# Print 256-dimensional vector of first embedded token
print(token_embeddings[0, 0])

tensor([ 4.9130e-01,  1.1239e+00,  1.4588e+00, -3.6530e-01, -4.0372e-02,
         1.9042e-01,  6.8024e-01, -8.6577e-01,  2.7644e-01,  6.3706e-01,
        -1.5947e+00, -2.6210e-02, -1.4804e+00, -1.0301e+00, -1.2018e+00,
         1.2016e+00,  1.8119e+00,  1.9705e+00,  1.6373e-01, -5.1866e-01,
         5.2069e-01, -1.0986e+00,  4.8871e-01,  4.3105e-01,  1.2858e+00,
        -1.9670e+00,  9.2041e-02,  4.5699e-01, -1.4753e+00, -4.4934e-01,
        -8.3517e-01, -1.1789e+00, -6.0601e-01, -5.3533e-01, -2.9543e-01,
         5.1638e-01,  4.2293e-01,  2.7947e-01, -2.4484e+00,  3.0050e-01,
         3.5847e-01,  7.4261e-01,  2.1654e-01, -9.2025e-01,  2.2604e-01,
        -8.0408e-01,  6.7722e-01,  8.9504e-01, -1.3292e+00, -1.6186e+00,
        -1.6062e+00, -8.2329e-01,  1.1738e-01, -7.6778e-01,  9.9370e-01,
         5.0167e-02,  7.0607e-02, -7.8336e-01,  4.4577e-01, -3.5213e-01,
         7.3934e-01,  3.3461e-01, -5.5481e-01, -3.8830e-01,  9.9484e-01,
         5.0054e-02, -1.5199e+00,  2.6316e-01, -1.6

**Positional Encoding**

To record token positional information about the tokens, use a second embedding layer of similar dimension to the token embedding layer, then add them together.

In [33]:
# Create position embedding layer
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [35]:
# Apply the positional embedding layer using a zero-based number sequence 
# which effectively records the token position index
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings)

tensor([[ 1.7375, -0.5620, -0.6303,  ..., -0.2277,  1.5748,  1.0345],
        [ 1.6423, -0.7201,  0.2062,  ...,  0.4118,  0.1498, -0.4628],
        [-0.4651, -0.7757,  0.5806,  ...,  1.4335, -0.4963,  0.8579],
        [-0.6754, -0.4628,  1.4323,  ...,  0.8139, -0.7088,  0.4827]],
       grad_fn=<EmbeddingBackward0>)


In [36]:
# Add the layers to combine token and positional embedding
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings)

tensor([[[ 2.2288,  0.5619,  0.8286,  ..., -0.6272, -0.2987,  0.8900],
         [ 2.0903, -0.4664, -0.0593,  ...,  0.9115, -1.0493, -1.6473],
         [-0.7158, -0.8304,  1.2494,  ...,  2.3952,  1.8773,  0.8051],
         [ 0.2703,  0.4029,  3.0514,  ...,  0.3595, -1.4548,  0.8310]],

        [[ 3.2835,  1.1749, -1.4150,  ..., -0.3281,  2.4332,  0.6924],
         [-0.2199, -0.9114, -0.1750,  ...,  1.5337, -0.1998,  0.1462],
         [ 1.5197, -1.4240,  0.4391,  ...,  1.0494, -1.4318,  2.3057],
         [ 0.2893,  0.8346, -0.1884,  ...,  1.9602,  0.8709,  0.8796]],

        [[ 0.9662,  0.0952, -0.4640,  ..., -1.0320,  1.6290,  1.7771],
         [ 2.4468, -0.2154,  1.4984,  ...,  1.8766,  0.5595, -0.1423],
         [-0.3856, -2.5393,  1.1556,  ...,  3.6157,  1.3267,  0.4944],
         [-0.2487, -0.5275,  2.0009,  ...,  0.2930,  0.5977,  1.3300]],

        ...,

        [[ 0.1219,  0.3991, -3.2740,  ..., -1.1921,  2.6637,  2.6728],
         [ 1.2438, -1.6436, -1.1101,  ..., -0.7464, -0.98